# 01 — QC & filtering

Quality control of public sarcoma 10x Visium data. We pull one sample from `data/raw/<sample>/` (downloaded by `scripts/fetch_public_data.py`), compute the usual per-spot QC metrics, and apply conservative cutoffs so downstream normalization isn't dominated by low-quality spots.

**Why this matters for sarcoma specifically.** Sarcoma sections are often FFPE archival material, which has more variable RNA integrity than fresh-frozen brain or breast tumor blocks. The dynamic range of total UMI per spot is wider, so applying a fixed cutoff blindly will throw away real stromal regions. We pick cutoffs visually and document them.

In [ ]:
# Parameters (overridable via papermill)
sample      = "GSE227469_angiosarcoma_01"
data_dir    = "data"
results_dir = "results"
min_counts  = 500
min_cells   = 10
max_pct_mt  = 25.0

In [ ]:
from pathlib import Path
import scanpy as sc
import matplotlib.pyplot as plt

sc.settings.set_figure_params(dpi=100, facecolor="white")
raw_dir = Path(data_dir) / "raw" / sample
out_dir = Path(results_dir) / "visium" / sample
out_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# Try scanpy's read_visium first; fall back to spatialdata-io if the
# downloaded archive isn't in the standard Space Ranger layout.
try:
    adata = sc.read_visium(raw_dir)
except Exception as exc:
    import spatialdata_io as sdio
    print(f"read_visium failed ({exc!s}); trying spatialdata-io.")
    sdata = sdio.visium(raw_dir)
    adata = sdata.tables["table"]

adata.var_names_make_unique()
adata

## QC metrics
We flag mitochondrial genes (MT-/mt- prefix) and compute `pct_counts_mt`, `n_genes_by_counts`, and `total_counts` per spot.

In [ ]:
adata.var["mt"] = adata.var_names.str.startswith(("MT-", "mt-"))
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], percent_top=None,
                           log1p=False, inplace=True)
adata.obs[["total_counts", "n_genes_by_counts", "pct_counts_mt"]].describe()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))
axes[0].hist(adata.obs["total_counts"], bins=60, color="#4c72b0")
axes[0].axvline(min_counts, color="red", linestyle="--", label=f"min={min_counts}")
axes[0].set_title("UMI / spot"); axes[0].legend()
axes[1].hist(adata.obs["n_genes_by_counts"], bins=60, color="#55a868")
axes[1].set_title("Genes / spot")
axes[2].hist(adata.obs["pct_counts_mt"], bins=60, color="#c44e52")
axes[2].axvline(max_pct_mt, color="black", linestyle="--", label=f"max={max_pct_mt}")
axes[2].set_title("% MT"); axes[2].legend()
fig.tight_layout()
fig.savefig(out_dir.parent.parent / "figures" / f"{sample}_qc_hist.pdf",
            bbox_inches="tight")

In [ ]:
# Visualize QC in spatial coordinates — a quick way to spot tissue edge artifacts.
sc.pl.spatial(adata, color=["total_counts", "n_genes_by_counts", "pct_counts_mt"],
              cmap="magma", ncols=3, size=1.4)

## Apply cutoffs
Conservative defaults: 500 UMI minimum, 10-cell gene minimum, 25% MT cap. Spots at the tissue edge often have artificially low UMI because they only partially overlap tissue; we accept that we'll lose a few of those.

In [ ]:
before = adata.n_obs
sc.pp.filter_cells(adata, min_counts=min_counts)
sc.pp.filter_genes(adata, min_cells=min_cells)
adata = adata[adata.obs["pct_counts_mt"] < max_pct_mt].copy()
print(f"Spots: {before} -> {adata.n_obs} (kept {adata.n_obs / before:.1%})")

In [ ]:
out_path = out_dir / "adata_qc.h5ad"
adata.write(out_path)
print("Wrote", out_path)

## Takeaways

- FFPE Visium spot UMIs are right-skewed; choose the lower cutoff visually rather than from a fixed quantile.
- Mitochondrial % is less interpretive in FFPE than in fresh tissue (RNA is partially degraded), so the 25% cap is generous on purpose.
- Spatial QC plots are the fastest way to spot tissue-mask issues — drift toward the tissue edge in `total_counts` usually means a misaligned hi-res image, not biology.